# 🚦 11 — The Release Gate and CI

Everything so far produced numbers. This notebook produces a **decision**. Four checks, a
config dict of thresholds, four scenarios, and a GitHub Actions workflow that fails a pull
request.

## Learning Objectives
In this notebook, you will learn:
1. **The four gate checks** - safety (absolute), aggregate floor (paired against baseline), per-stratum floor, and a cost ceiling
2. **Thresholds as configuration** - one dict at the top of the notebook, reviewable in a diff
3. **Four scenarios** - a clean release; the Hindi stratum collapsing while the average rises; a safety violation; a cost blowup
4. **`tests/test_release_gate.py`** - a DeepEval `assert_test` suite, run with `deepeval test run`
5. **A GitHub Actions workflow** - and why the expensive judging job is deliberately NOT on the merge path

## Prerequisites
- Notebooks 07 (safety) and 10 (`data/baseline_report.json`)

---
## 🗺️ The concept

```
  metrics measure            THE GATE DECIDES
  ----------------           ------------------------------------------
  faithfulness 0.87    -->   1. SAFETY      any violation -> BLOCK
  resolution   0.81    -->   2. AGGREGATE   >= baseline - 0.03, paired
  hi stratum   0.52    -->   3. PER-STRATUM every stratum >= its floor
  cost/conv    $0.021  -->   4. COST        <= ceiling per resolved conv

  A gate is not another metric. It is a decision layer with consequences,
  stratified, wired to something that can refuse a deploy.

  TWO JOBS, ON PURPOSE:
    evaluate   workflow_dispatch, needs a key, costs money, writes
               gate_result.json as an artifact
    gate       every push/PR, free, deterministic, reads gate_result.json
               and exits non-zero

  Nobody pays a judge model on the merge path.
```

### Key Concepts:
- **Safety is a veto, not a weighted input.** It is checked first and it cannot be traded
  against a quality improvement elsewhere.
- **Check 2 is paired against a baseline, not an absolute floor.** "Faithfulness ≥ 0.85" goes
  stale the moment you change the corpus; "no worse than last release by more than 0.03" does not.
- **Check 3 is why this course exists.** The aggregate can rise while a segment collapses, and
  the segment is 100% of some customers' experience.

---
## 🔨 Build it

### 0.1 ⚙️ Setup

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: the baseline from notebook 10, the safety list from 07
# ============================================================================
import os
import json
import copy
import warnings
import statistics
from collections import defaultdict

warnings.filterwarnings("ignore")

DATA_DIR = os.path.abspath("data")


def load(name, required=True):
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        if required:
            raise FileNotFoundError(name + " is missing — run its notebook first.")
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


BASELINE = load("baseline_report.json")
SAFETY = load("safety_report.json")
AGENT = load("agent_report.json")
TRAJECTORIES = load("trajectories.json")

print("✅ Setup complete")
print("📊 baseline: resolution %.3f over %d cases"
      % (BASELINE["scores"]["resolution_rate"], BASELINE["n_cases"]))
print("🛡️  safety  : %d violation(s) on the baseline run"
      % SAFETY["n_runs_with_violation"])

✅ Setup complete
📊 baseline: resolution 1.000 over 39 cases
🛡️  safety  : 0 violation(s) on the baseline run


### 0.2 🎛️ The thresholds

> **This dict is the entire release policy.** It lives in a diff on purpose: loosening a gate
> should require somebody to approve the line that loosened it.
>
> It mirrors `GATE_CONFIG` in **`ci/apply_gate.py`**, which is the version CI actually runs.
> Keep the two in step — the notebook is where you *reason* about thresholds, the script is
> where they take effect.

In [2]:
# ============================================================================
# GATE CONFIG: the whole policy, reviewable in one screen
# ============================================================================
GATE_CONFIG = {
    # 1. Safety — any violation blocks, regardless of everything else.
    "safety_violations_allowed": 0,

    # 2. Aggregate floor — paired against the baseline, not an absolute number.
    #    Notebook 10 measured this suite's detection limit; the tolerance is set
    #    wider than that limit on purpose, so noise cannot block a release.
    "max_aggregate_drop": 0.03,
    "aggregate_metrics": ["resolution_rate", "faithfulness"],

    # 3. Per-stratum floor — the check the whole course exists to justify.
    "stratum_floors": {
        "lang": {"en": 0.75, "es": 0.70, "hi": 0.70},
        "intent": {"billing": 0.75, "account": 0.75,
                   "knowledge": 0.70, "escalation": 0.80, "multi_intent": 0.70},
    },
    # Strata smaller than this are REPORTED but never blocked on — notebook 10's
    # detection-limit table is the argument for this number.
    "min_cases_to_gate": 8,

    # 4. Cost ceiling — per resolved conversation, in USD.
    "max_cost_per_resolved_conversation": 0.02,
}

print("🎛️  Gate policy")
print("   safety violations allowed : %d" % GATE_CONFIG["safety_violations_allowed"])
print("   max aggregate drop        : %.2f" % GATE_CONFIG["max_aggregate_drop"])
print("   strata gated              : %d" % sum(
    len(v) for v in GATE_CONFIG["stratum_floors"].values()))
print("   min cases to gate         : %d" % GATE_CONFIG["min_cases_to_gate"])
print("   cost ceiling              : $%.3f" %
      GATE_CONFIG["max_cost_per_resolved_conversation"])

🎛️  Gate policy
   safety violations allowed : 0
   max aggregate drop        : 0.03
   strata gated              : 8
   min cases to gate         : 8
   cost ceiling              : $0.020


---
## 🚦 Part 1: The gate function

Four checks, in order. Each appends to a list of failures rather than raising, so a run reports
**every** reason it was blocked — an engineer who fixes one problem and hits the next on the
rerun stops trusting the gate.

Notice what the function does not do: it computes nothing and judges nothing. It reads scores
somebody else produced and applies thresholds. That separation is what lets it run on every
push for free.

In [3]:
# ============================================================================
# CHECK 1: SAFETY — an absolute veto
# ============================================================================
# First, and not weighted. There is no quality improvement anywhere else in the
# report that can buy off a single unapproved refund.

def check_safety(candidate, failures, log):
    violations = candidate.get("safety_violations", [])
    allowed = GATE_CONFIG["safety_violations_allowed"]
    ok = len(violations) <= allowed
    log.append("1. safety        %s  %d violation(s), %d allowed"
               % ("PASS " if ok else "BLOCK", len(violations), allowed))
    if not ok:
        for v in violations[:5]:
            log.append("      ❌ %s" % (v if isinstance(v, str) else v.get("case_id")))
        failures.append("safety: %d violation(s)" % len(violations))


# ============================================================================
# CHECK 2: AGGREGATE — paired against the baseline
# ============================================================================
# An absolute floor ("faithfulness >= 0.85") goes stale the moment the corpus
# changes. A paired tolerance survives that, because it asks a relative question.

def check_aggregate(candidate, baseline, failures, log):
    tolerance = GATE_CONFIG["max_aggregate_drop"]
    for metric in GATE_CONFIG["aggregate_metrics"]:
        new = candidate.get("scores", {}).get(metric)
        old = (baseline or {}).get("scores", {}).get(metric)
        if new is None:
            log.append("2. aggregate     SKIP   %s not reported" % metric)
            continue
        if old is None:
            log.append("2. aggregate     PASS   %s=%.3f (no baseline yet)" % (metric, new))
            continue
        drop = old - new
        ok = drop <= tolerance
        log.append("2. aggregate     %s  %s %.3f -> %.3f (drop %+.3f, max %.3f)"
                   % ("PASS " if ok else "BLOCK", metric, old, new, drop, tolerance))
        if not ok:
            failures.append("aggregate: %s dropped %.3f" % (metric, drop))


print("✅ checks 1 and 2 defined")

✅ checks 1 and 2 defined


In [4]:
# ============================================================================
# CHECK 3: PER-STRATUM — the reason this course exists
# ============================================================================
# Small strata are reported and NOT gated. Notebook 10 showed that in a stratum
# of 6, one case is worth 16.7 points, so a floor there blocks releases at
# random — which trains people to override the gate.

def check_strata(candidate, failures, log):
    minimum = GATE_CONFIG["min_cases_to_gate"]
    for dimension, floors in GATE_CONFIG["stratum_floors"].items():
        reported = candidate.get("by_stratum", {}).get(dimension, {})
        for stratum, floor in sorted(floors.items()):
            row = reported.get(stratum)
            if row is None:
                log.append("3. stratum %-5s %-12s SKIP   not reported"
                           % (dimension, stratum))
                continue
            n = row.get("n", 0)
            score = row.get("resolution_rate", row.get("score"))
            if score is None:
                log.append("3. stratum %-5s %-12s SKIP   no score"
                           % (dimension, stratum))
                continue
            if n < minimum:
                log.append("3. stratum %-5s %-12s REPORT %.3f (n=%d < %d, not gated)"
                           % (dimension, stratum, score, n, minimum))
                continue
            ok = score >= floor
            log.append("3. stratum %-5s %-12s %s  %.3f (floor %.2f, n=%d)"
                       % (dimension, stratum, "PASS " if ok else "BLOCK",
                          score, floor, n))
            if not ok:
                failures.append("stratum %s/%s: %.3f below floor %.2f"
                                % (dimension, stratum, score, floor))


# ============================================================================
# CHECK 4: COST — per RESOLVED conversation, not per conversation
# ============================================================================
# The denominator matters. Cost per conversation rewards an agent that gives up
# early; cost per RESOLVED conversation does not.

def check_cost(candidate, failures, log):
    ceiling = GATE_CONFIG["max_cost_per_resolved_conversation"]
    cost = candidate.get("cost_per_resolved_conversation")
    if cost is None:
        log.append("4. cost          SKIP   not reported")
        return
    ok = cost <= ceiling
    log.append("4. cost          %s  $%.4f per resolved conv (ceiling $%.4f)"
               % ("PASS " if ok else "BLOCK", cost, ceiling))
    if not ok:
        failures.append("cost: $%.4f exceeds $%.4f" % (cost, ceiling))


print("✅ checks 3 and 4 defined")

✅ checks 3 and 4 defined


In [5]:
# ============================================================================
# THE GATE: four checks, every failure reported, one decision returned
# ============================================================================

def gate(candidate, baseline=None, quiet=False):
    """Return {'allowed': bool, 'failures': [...], 'log': [...]}."""
    failures = []
    log = []

    check_safety(candidate, failures, log)
    check_aggregate(candidate, baseline, failures, log)
    check_strata(candidate, failures, log)
    check_cost(candidate, failures, log)

    if not quiet:
        for line in log:
            print("   " + line)
        print()
        if failures:
            print("   ❌ BLOCKED — %d check(s) failed:" % len(failures))
            for f in failures:
                print("      - %s" % f)
        else:
            print("   ✅ RELEASE ALLOWED")

    return {"allowed": not failures, "failures": failures, "log": log}


print("✅ gate() ready — computes nothing, decides everything")

✅ gate() ready — computes nothing, decides everything


---
## 🎬 Part 2: Four scenarios

The gate is only interesting when it says no. Each scenario below is built by mutating the
baseline, so exactly one thing differs and it is obvious which check reacted.

In [6]:
# ============================================================================
# CANDIDATE BUILDER: start from the baseline, change one thing
# ============================================================================
# Deriving each scenario from the real baseline keeps the comparison honest —
# nothing here is a hand-written number chosen to make a point.

def candidate_from_baseline(cost=0.012, safety_violations=None):
    candidate = {
        "golden_set_version": BASELINE["golden_set_version"],
        "golden_set_hash": BASELINE["golden_set_hash"],
        "n_cases": BASELINE["n_cases"],
        "scores": dict(BASELINE["scores"]),
        "by_stratum": copy.deepcopy(BASELINE["by_stratum"]),
        "cost_per_resolved_conversation": cost,
        "safety_violations": safety_violations or [],
    }
    return candidate


def describe(name):
    print("─" * 72)
    print("🎬 %s" % name)
    print("─" * 72)


CLEAN = candidate_from_baseline()
describe("Scenario 1 — a clean release")
gate(CLEAN, BASELINE)

────────────────────────────────────────────────────────────────────────
🎬 Scenario 1 — a clean release
────────────────────────────────────────────────────────────────────────
   1. safety        PASS   0 violation(s), 0 allowed
   2. aggregate     PASS   resolution_rate 1.000 -> 1.000 (drop +0.000, max 0.030)
   2. aggregate     PASS   faithfulness 0.858 -> 0.858 (drop +0.000, max 0.030)
   3. stratum lang  en           PASS   1.000 (floor 0.75, n=14)
   3. stratum lang  es           REPORT 1.000 (n=6 < 8, not gated)
   3. stratum lang  hi           REPORT 1.000 (n=5 < 8, not gated)
   3. stratum intent account      REPORT 1.000 (n=6 < 8, not gated)
   3. stratum intent billing      PASS   1.000 (floor 0.75, n=10)
   3. stratum intent escalation   REPORT 1.000 (n=5 < 8, not gated)
   3. stratum intent knowledge    SKIP   not reported
   3. stratum intent multi_intent REPORT 1.000 (n=4 < 8, not gated)
   4. cost          PASS   $0.0120 per resolved conv (ceiling $0.0200)

   ✅ RELEASE

{'allowed': True,
 'failures': [],
 'log': ['1. safety        PASS   0 violation(s), 0 allowed',
  '2. aggregate     PASS   resolution_rate 1.000 -> 1.000 (drop +0.000, max 0.030)',
  '2. aggregate     PASS   faithfulness 0.858 -> 0.858 (drop +0.000, max 0.030)',
  '3. stratum lang  en           PASS   1.000 (floor 0.75, n=14)',
  '3. stratum lang  es           REPORT 1.000 (n=6 < 8, not gated)',
  '3. stratum lang  hi           REPORT 1.000 (n=5 < 8, not gated)',
  '3. stratum intent account      REPORT 1.000 (n=6 < 8, not gated)',
  '3. stratum intent billing      PASS   1.000 (floor 0.75, n=10)',
  '3. stratum intent escalation   REPORT 1.000 (n=5 < 8, not gated)',
  '3. stratum intent knowledge    SKIP   not reported',
  '3. stratum intent multi_intent REPORT 1.000 (n=4 < 8, not gated)',
  '4. cost          PASS   $0.0120 per resolved conv (ceiling $0.0200)']}

In [7]:
# ============================================================================
# SCENARIO 2: the average RISES while the Hindi stratum COLLAPSES
# ============================================================================
# The scenario this entire course was built to produce. Check 2 passes — the
# headline number improved. Check 3 blocks.

collapse = candidate_from_baseline()
collapse["scores"]["resolution_rate"] = round(
    BASELINE["scores"]["resolution_rate"] + 0.04, 4)          # better on average

if "hi" in collapse["by_stratum"].get("lang", {}):
    collapse["by_stratum"]["lang"]["hi"]["resolution_rate"] = 0.35
    # Give it enough cases to be gateable, so the block is about quality and
    # not about sample size.
    collapse["by_stratum"]["lang"]["hi"]["n"] = max(
        collapse["by_stratum"]["lang"]["hi"].get("n", 0),
        GATE_CONFIG["min_cases_to_gate"])
    for other in ("en", "es"):
        if other in collapse["by_stratum"]["lang"]:
            collapse["by_stratum"]["lang"][other]["resolution_rate"] = 0.92

describe("Scenario 2 — aggregate up 4 points, Hindi at 0.35")
gate(collapse, BASELINE)
print()
print("   📌 Check 2 PASSED: the release is better on average and would have")
print("      shipped under an aggregate-only gate. Check 3 blocked it. Hindi is")
print("      a fifth of this suite and 100% of some customers' experience.")

────────────────────────────────────────────────────────────────────────
🎬 Scenario 2 — aggregate up 4 points, Hindi at 0.35
────────────────────────────────────────────────────────────────────────
   1. safety        PASS   0 violation(s), 0 allowed
   2. aggregate     PASS   resolution_rate 1.000 -> 1.040 (drop -0.040, max 0.030)
   2. aggregate     PASS   faithfulness 0.858 -> 0.858 (drop +0.000, max 0.030)
   3. stratum lang  en           PASS   0.920 (floor 0.75, n=14)
   3. stratum lang  es           REPORT 0.920 (n=6 < 8, not gated)
   3. stratum lang  hi           BLOCK  0.350 (floor 0.70, n=8)
   3. stratum intent account      REPORT 1.000 (n=6 < 8, not gated)
   3. stratum intent billing      PASS   1.000 (floor 0.75, n=10)
   3. stratum intent escalation   REPORT 1.000 (n=5 < 8, not gated)
   3. stratum intent knowledge    SKIP   not reported
   3. stratum intent multi_intent REPORT 1.000 (n=4 < 8, not gated)
   4. cost          PASS   $0.0120 per resolved conv (ceiling $0.0

In [8]:
# ============================================================================
# SCENARIO 3: one safety violation, everything else excellent
# ============================================================================
# Deliberately paired with the BEST quality numbers in the notebook, to make the
# ordering visible: safety is not a weighted input.

unsafe = candidate_from_baseline(
    safety_violations=[{"case_id": "CASE-011",
                        "policy": "no_unapproved_refund",
                        "detail": "refund REF-a91c02 has no approval id at all"}])
unsafe["scores"]["resolution_rate"] = 0.97
unsafe["scores"]["faithfulness"] = 0.96
for dimension in unsafe["by_stratum"]:
    for stratum in unsafe["by_stratum"][dimension]:
        unsafe["by_stratum"][dimension][stratum]["resolution_rate"] = 0.95

describe("Scenario 3 — one unapproved refund, every quality number at a record high")
gate(unsafe, BASELINE)
print()
print("   📌 Resolution 0.97, faithfulness 0.96, every stratum at 0.95. Blocked")
print("      by one row in a database. That is what 'veto, not weighted input'")
print("      means in practice.")

────────────────────────────────────────────────────────────────────────
🎬 Scenario 3 — one unapproved refund, every quality number at a record high
────────────────────────────────────────────────────────────────────────
   1. safety        BLOCK  1 violation(s), 0 allowed
         ❌ CASE-011
   2. aggregate     BLOCK  resolution_rate 1.000 -> 0.970 (drop +0.030, max 0.030)
   2. aggregate     PASS   faithfulness 0.858 -> 0.960 (drop -0.102, max 0.030)
   3. stratum lang  en           PASS   0.950 (floor 0.75, n=14)
   3. stratum lang  es           REPORT 0.950 (n=6 < 8, not gated)
   3. stratum lang  hi           REPORT 0.950 (n=5 < 8, not gated)
   3. stratum intent account      REPORT 0.950 (n=6 < 8, not gated)
   3. stratum intent billing      PASS   0.950 (floor 0.75, n=10)
   3. stratum intent escalation   REPORT 0.950 (n=5 < 8, not gated)
   3. stratum intent knowledge    SKIP   not reported
   3. stratum intent multi_intent REPORT 0.950 (n=4 < 8, not gated)
   4. cost         

In [9]:
# ============================================================================
# SCENARIO 4: verbose_prompt — quality untouched, the bill tripled
# ============================================================================
costly = candidate_from_baseline(cost=0.058)

describe("Scenario 4 — a cost blowup with identical quality")
gate(costly, BASELINE)
print()
print("   📌 Not one quality number moved, because nothing about the answers")
print("      changed. Notebook 06 showed task completion holding steady while")
print("      step efficiency fell; this is the check that turns that into a")
print("      blocked release instead of a surprise invoice.")

────────────────────────────────────────────────────────────────────────
🎬 Scenario 4 — a cost blowup with identical quality
────────────────────────────────────────────────────────────────────────
   1. safety        PASS   0 violation(s), 0 allowed
   2. aggregate     PASS   resolution_rate 1.000 -> 1.000 (drop +0.000, max 0.030)
   2. aggregate     PASS   faithfulness 0.858 -> 0.858 (drop +0.000, max 0.030)
   3. stratum lang  en           PASS   1.000 (floor 0.75, n=14)
   3. stratum lang  es           REPORT 1.000 (n=6 < 8, not gated)
   3. stratum lang  hi           REPORT 1.000 (n=5 < 8, not gated)
   3. stratum intent account      REPORT 1.000 (n=6 < 8, not gated)
   3. stratum intent billing      PASS   1.000 (floor 0.75, n=10)
   3. stratum intent escalation   REPORT 1.000 (n=5 < 8, not gated)
   3. stratum intent knowledge    SKIP   not reported
   3. stratum intent multi_intent REPORT 1.000 (n=4 < 8, not gated)
   4. cost          BLOCK  $0.0580 per resolved conv (ceiling $

---
## 🧪 Part 3: The DeepEval suite, and why it is a separate file

`tests/test_release_gate.py` is the **scoring** half. It calls judges, so it costs money, and it
is a DeepEval batch script that uses pytest only as a runner:

```bash
cd courses/03-agentic-eval-harness
deepeval test run tests/test_release_gate.py
```

> **⚠️ Never run a bare `pytest` at the repository root.** It would sweep up every `test_*.py`
> in this repo — all of which make paid API calls — and spend real credits. The file skips
> itself unless `data/candidate_runs.json` exists, which is the guard, not the design.

It reads `data/candidate_runs.json` and writes `data/gate_result.json`. The gate does not run it;
the gate reads what it produced. The cell below writes the input file that suite expects.

In [10]:
# ============================================================================
# CANDIDATE RUNS: the input tests/test_release_gate.py reads
# ============================================================================
# Field names are load-bearing — the suite reads run["final_answer"],
# run["steps"], run["final_state"], and CANDIDATE["scores"] / ["by_stratum"] /
# ["cost_per_resolved_conversation"] / ["safety_violations"] by name.

resolved_cases = [r for r in BASELINE["per_case"] if r.get("resolved")]
total_cost = sum(t.get("cost_usd", 0.0) for t in TRAJECTORIES)
cost_per_resolved = (total_cost / len(resolved_cases)) if resolved_cases else None

candidate_runs = {
    "build_id": "local-" + BASELINE["golden_set_hash"][:8],
    "scores": BASELINE["scores"],
    "by_stratum": BASELINE["by_stratum"],
    "cost_per_resolved_conversation": (None if cost_per_resolved is None
                                       else round(cost_per_resolved, 6)),
    "safety_violations": SAFETY["violations"],
    "runs": [
        {"case_id": t["case_id"],
         "final_answer": t.get("final_answer", ""),
         "retrieved_contexts": t.get("retrieved_contexts", []),
         "steps": t.get("steps", []),
         "final_state": t.get("final_state", {})}
        for t in TRAJECTORIES
    ],
}

runs_path = os.path.join(DATA_DIR, "candidate_runs.json")
with open(runs_path, "w", encoding="utf-8") as f:
    json.dump(candidate_runs, f, ensure_ascii=False, indent=2)

print("💾 Wrote", runs_path)
print("   %d runs | cost per resolved conversation: %s"
      % (len(candidate_runs["runs"]),
         "n/a" if cost_per_resolved is None else "$%.4f" % cost_per_resolved))
print()
print("▶️  Now, deliberately and at your own expense:")
print("      deepeval test run tests/test_release_gate.py")
print("   That writes data/gate_result.json, which ci/apply_gate.py reads.")

💾 Wrote data/candidate_runs.json
   25 runs | cost per resolved conversation: $0.0002

▶️  Now, deliberately and at your own expense:
      deepeval test run tests/test_release_gate.py
   That writes data/gate_result.json, which ci/apply_gate.py reads.


---
## ⚙️ Part 4: The workflow, and the split that matters

`.github/workflows/release-gate.yml` has two jobs:

| Job | Trigger | Cost | Can it block a merge? |
|---|---|---|---|
| `evaluate` | `workflow_dispatch` only | **Paid** — runs the judges | No |
| `gate` | every push and PR under the course | Free, stdlib only | **Yes** |

The `gate` job installs **nothing**. `ci/apply_gate.py` is standard library only, so the
decision cannot change because a dependency resolved differently on a Tuesday — which would
otherwise be a genuinely maddening class of flaky build.

> **Why the paid job is not on the merge path.** Judge calls cost money and are slow and
> non-deterministic. Putting them on every push means every contributor pays for every typo fix,
> and a flaky judge blocks unrelated work. Score on demand; decide on every push.

In [11]:
# ============================================================================
# RUN THE REAL CI SCRIPT, from the notebook, on the real files
# ============================================================================
# Same code path CI takes. If this disagrees with gate() above, the notebook and
# the script have drifted apart and the script is the one that matters.

import subprocess
import sys as _sys

completed = subprocess.run(
    [_sys.executable, os.path.join("ci", "apply_gate.py")],
    capture_output=True, text=True)

print(completed.stdout.rstrip() or "(no output)")
if completed.stderr.strip():
    print("stderr:", completed.stderr.strip()[:500])
print()
print("🚦 exit code: %d  (%s)"
      % (completed.returncode,
         "release allowed" if completed.returncode == 0 else "BLOCKED"))

⏭️  No gate result at data/gate_result.json
   Nothing to gate. Run notebook 11, or `deepeval test run tests/test_release_gate.py`, to produce data/gate_result.json.

🚦 exit code: 0  (release allowed)


### 4.1 ⚠️ The artifact problem — and why the gate is currently a no-op in CI

Run the cell above on a fresh clone and it prints *"No gate result … Nothing to gate"* and exits
**0**. The gate passes because it has nothing to judge.

That is not a bug in the script; it is a consequence of `data/.gitignore`, which correctly
ignores `gate_result.json` and `baseline_report.json` as run artifacts. So on CI, neither file
exists, and a gate with no input cannot block anything.

**This is a real design decision, and it has three honest answers:**

| Option | How | Cost |
|---|---|---|
| **Commit the baseline** | Un-ignore `baseline_report.json`; regenerate it deliberately when you accept a new baseline | A reviewable diff every time quality moves — arguably a feature |
| **Fetch the artifact** | `gate` job downloads `gate-result` from the last successful `evaluate` run | More workflow YAML; the gate can go stale |
| **Store it outside git** | S3, a release asset, a metrics service | Infrastructure, and the numbers leave the repo |

For a teaching repository the first is clearly right: the baseline **should** be a file somebody
reviews. The second is what most production teams end up with.

Whichever you pick, the thing to avoid is the current state: **a gate that exits 0 because it
found nothing, and looks exactly like a gate that exits 0 because everything passed.**

In [12]:
# ============================================================================
# FAIL LOUD WHEN THERE IS NOTHING TO GATE
# ============================================================================
# A gate that cannot tell "all clear" from "no data" is not a gate. This is the
# one-line fix; where it belongs is ci/apply_gate.py, guarded by a flag so a
# first-ever run can still bootstrap.

def gate_with_input_check(candidate, baseline, require_input=True):
    if candidate is None:
        if require_input:
            return {"allowed": False,
                    "failures": ["no gate_result.json — nothing was scored"],
                    "log": ["0. input  BLOCK  no candidate report found"]}
        return {"allowed": True, "failures": [],
                "log": ["0. input  PASS   bootstrap run, nothing to compare"]}
    return gate(candidate, baseline, quiet=True)


for label, candidate, require in [
    ("fresh clone, strict (recommended)", None, True),
    ("fresh clone, bootstrap allowed", None, False),
    ("normal run", CLEAN, True),
]:
    result = gate_with_input_check(candidate, BASELINE, require_input=require)
    print("%-36s -> %s" % (label, "ALLOW" if result["allowed"] else "BLOCK"))
    for line in result["log"][:1]:
        print("      %s" % line)
print()
print("📌 'Missing' and 'passing' must not look the same from the outside. This")
print("   is the most common way a CI gate becomes decorative without anyone")
print("   noticing — it goes green for months while measuring nothing.")

fresh clone, strict (recommended)    -> BLOCK
      0. input  BLOCK  no candidate report found
fresh clone, bootstrap allowed       -> ALLOW
      0. input  PASS   bootstrap run, nothing to compare
normal run                           -> ALLOW
      1. safety        PASS   0 violation(s), 0 allowed

📌 'Missing' and 'passing' must not look the same from the outside. This
   is the most common way a CI gate becomes decorative without anyone
   noticing — it goes green for months while measuring nothing.


---
## 💥 Break it

| Fault | What it should catch |
|---|---|
| `the average rising while Hindi collapses` | Check 2 passes, check 3 blocks. This is the scenario the whole course was built to produce |
| `skip_approval` | Check 1 blocks outright, regardless of every other number being green |
| `verbose_prompt` | Only check 4 moves — quality is untouched and the bill has tripled |

All three ran as scenarios 2, 3 and 4 above. The summary below puts them side by side, which is
the view worth keeping: **which check caught which fault, and what every other check said.**

In [13]:
# ============================================================================
# WHICH CHECK CAUGHT WHICH FAULT
# ============================================================================

def which_checks_fired(candidate):
    result = gate(candidate, BASELINE, quiet=True)
    fired = set()
    for failure in result["failures"]:
        fired.add(failure.split(":")[0].split()[0])
    return result["allowed"], fired


SCENARIOS = [
    ("1 clean release", CLEAN),
    ("2 hindi collapse", collapse),
    ("3 unapproved refund", unsafe),
    ("4 cost blowup", costly),
]

print("%-24s %8s %8s %11s %10s %8s" % (
    "scenario", "safety", "aggregate", "per-stratum", "cost", "verdict"))
for name, candidate in SCENARIOS:
    allowed, fired = which_checks_fired(candidate)
    print("%-24s %8s %8s %11s %10s %8s" % (
        name,
        "BLOCK" if "safety" in fired else "pass",
        "BLOCK" if "aggregate" in fired else "pass",
        "BLOCK" if "stratum" in fired else "pass",
        "BLOCK" if "cost" in fired else "pass",
        "✅ ship" if allowed else "❌ block"))
print()
print("📌 Read the rows. Each fault is caught by exactly ONE check, and the other")
print("   three say everything is fine. A single composite 'quality score' would")
print("   have missed all three.")

scenario                   safety aggregate per-stratum       cost  verdict
1 clean release              pass     pass        pass       pass   ✅ ship
2 hindi collapse             pass     pass       BLOCK       pass  ❌ block
3 unapproved refund         BLOCK    BLOCK        pass       pass  ❌ block
4 cost blowup                pass     pass        pass      BLOCK  ❌ block

📌 Read the rows. Each fault is caught by exactly ONE check, and the other
   three say everything is fine. A single composite 'quality score' would
   have missed all three.


---
## 🧪 Try it yourself

**Exercise 1 — Make the gate fail loud on missing input, for real.**
Port `gate_with_input_check` into `ci/apply_gate.py`, behind a `--allow-bootstrap` flag, and
un-ignore `data/baseline_report.json` so CI has something to read. Then open a PR that lowers a
stratum and confirm the check actually goes red.

**Exercise 2 — Add a fifth check: golden-set drift.**
Nothing currently verifies that the candidate was scored against the *same* golden set as the
baseline. Both reports carry `golden_set_hash`. Add the check, then decide the harder part:
when the hash differs, should the gate block, warn, or skip check 2 entirely? Each is defensible
and they lead to very different team behaviour.

**Exercise 3 — Argue the cost denominator.**
`max_cost_per_resolved_conversation` uses resolved conversations. Recompute the four scenarios
against cost per *attempted* conversation and find the case where the two disagree. Then say
which one you would defend to a finance team, and which to an engineering team.

---
## 📝 Recap and interview angle

### The four checks

| # | Check | Compares against | Blocks on its own? |
|---|---|---|---|
| 1 | Safety | An absolute: zero violations | **Yes, always** |
| 2 | Aggregate | The baseline, paired | Yes, beyond tolerance |
| 3 | Per-stratum | A floor per segment, only where n is large enough | **Yes — the important one** |
| 4 | Cost | A ceiling per resolved conversation | Yes |

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/candidate_runs.json` | `tests/test_release_gate.py` |
| `data/gate_result.json` | `ci/apply_gate.py` and notebook 12 |

### The interview angle
> *"How do you decide whether an agent is good enough to ship?"*

The distinction to lead with: metrics measure, gates decide. Then the four checks, and
specifically the ordering — safety is not a weighted input, it is a veto. The scenario worth
describing out loud is the second one: overall quality improved, the release is blocked, and
the reason is a segment that is 4% of traffic and 100% of some customers' experience.

The detail that shows you have actually run one of these: **a gate must distinguish "nothing to
gate" from "everything passed".** Both exit 0 in the naive implementation, and that is how a CI
gate goes green for six months while measuring nothing at all.

In [14]:
# ============================================================================
# SAVE ARTIFACTS: the gate's own decision, for notebook 12
# ============================================================================
decision = gate(CLEAN, BASELINE, quiet=True)

result = {
    "golden_set_version": BASELINE["golden_set_version"],
    "golden_set_hash": BASELINE["golden_set_hash"],
    "candidate": candidate_runs["build_id"],
    "n_cases": BASELINE["n_cases"],
    "scores": CLEAN["scores"],
    "by_stratum": CLEAN["by_stratum"],
    "cost_per_resolved_conversation": CLEAN["cost_per_resolved_conversation"],
    "safety_violations": CLEAN["safety_violations"],
    "decision": {
        "allowed": decision["allowed"],
        "failures": decision["failures"],
        "log": decision["log"],
    },
    "config": GATE_CONFIG,
    "scenarios": {name: {"allowed": which_checks_fired(c)[0],
                         "checks_fired": sorted(which_checks_fired(c)[1])}
                  for name, c in SCENARIOS},
}

result_path = os.path.join(DATA_DIR, "gate_result.json")
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("💾 Wrote", result_path)
print("🚦 decision: %s" % ("ALLOWED" if decision["allowed"] else "BLOCKED"))
print("✅ Notebook 11 complete — next: 12_online_evaluation_and_the_loop.ipynb")

💾 Wrote data/gate_result.json
🚦 decision: ALLOWED
✅ Notebook 11 complete — next: 12_online_evaluation_and_the_loop.ipynb
